# 03 · EDA y análisis

### Enfoque

Este notebook **no recorre variable por variable**. Tiene una tesis y la defiende:

> Las tres relaciones más fuertes del dataset son **no monótonas**, y en las tres la correlación
> engaña. Una selección de variables basada en correlación habría descartado el mejor predictor.

El motivo de no hacer el recorrido exhaustivo no es la prisa: es que **un gráfico por variable es
univariante**, y los dos hallazgos más valiosos del proyecto —el confusor de país y el de la edad
en las filas incoherentes— no habrían aparecido en ninguno. Las relaciones que importan casi nunca
están en una variable sola.

### Pero la tesis no puede decidir la cobertura

Estructurar por tesis tiene un riesgo real: mirar solo donde confirma. Se controla con tres capas
**independientes** del argumento:

1. **Matriz de cobertura** — cada variable debe aparecer al menos una vez contra el objetivo
   *por grupos*, no por correlación. Es una lista de comprobación mecánica, inmune al sesgo.
2. **Cruces con los predictores dominantes** — cada variable contra `age`, `num_of_products`,
   `is_active_member` y `geography`. Si algo aparentemente inútil se comporta distinto dentro de
   algún nivel, salta ahí.
3. **El modelo como auditor** — un Random Forest recorre el espacio de interacciones sin que nadie
   le indique dónde mirar. Si en el notebook 04 da importancia alta a algo que aquí descartamos, o
   parte por una variable que no discutimos, **se nos escapó** y se vuelve a este notebook.

La tercera capa es la que de verdad responde a la pregunta *«¿cómo sé si se me escapó algo?»*. Por
eso el EDA no necesita ser exhaustivo: necesita ser sistemático y tener un mecanismo de vuelta
atrás.

## Entorno

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

plt.rcParams.update({
    "figure.dpi": 110,
    "figure.figsize": (9, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

CATALOG      = "bank_churn"
SILVER_TABLE = f"{CATALOG}.silver.bank_customers_clean"

df = spark.table(SILVER_TABLE).toPandas()
df["exited"] = df["exited"].astype(bool)

TASA_BASE = df["exited"].mean()
N         = len(df)
N_CHURN   = int(df["exited"].sum())

print(f"{N:,} clientes · {N_CHURN:,} abandonos · tasa base {TASA_BASE:.2%}")
print("columnas:", list(df.columns))

10,000 clientes · 2,037 abandonos · tasa base 20.37%
columnas: ['customer_id', 'credit_score', 'geography', 'gender', 'age', 'tenure', 'balance', 'num_of_products', 'has_cr_card', 'is_active_member', 'estimated_salary', 'exited', 'balance_zero', 'age_group', 'products_group', 'credit_score_band', '_silver_at']


---

## A · Matriz de cobertura

Antes de argumentar nada, comprobar que ninguna variable se quedó sin examinar. La tabla se
declara **antes** de mirar los resultados, para que la tesis no pueda influir en qué se revisa.

| Variable | Estado previo | Dónde se cubre |
|---|---|---|
| `age` | analizada — U invertida | sección B |
| `num_of_products` | analizada — 7,6% a 100% | sección B |
| `balance` / `balance_zero` | analizada — estado distinto, no nulo | sección B, sección C |
| `is_active_member` | analizada — 26,9% vs 14,3% | sección D |
| `geography` | analizada — Alemania inexplicada | sección C |
| `gender` | analizada — +8,6 pp | sección C |
| `tenure` | analizada — ruido | sección E |
| `estimated_salary` | analizada — ruido uniforme | sección E |
| `credit_score` | analizada — plano en cinco tramos | sección E |
| **`has_cr_card`** | **nunca examinada** | **sección A · hueco 1** |

### Tres huecos detectados

1. **`has_cr_card` no se ha mirado ni una vez.** La intuición dice que tener tarjeta es casi
   universal y no discriminará — pero eso es una suposición, del mismo tipo que ya falló dos veces
   en este proyecto.

2. **No sabemos si el efecto de la actividad es igual a todas las edades.** Es *la* variable
   accionable: si los inactivos de 50-59 abandonan al 70% y los de 30-39 al 15%, la recomendación
   prescriptiva cambia por completo.

3. **Nunca se midió la colinealidad entre predictores.** Anotamos que `balance_zero` está enredada
   con `geography` y los productos con el saldo, pero solo hemos mirado cada variable *contra el
   objetivo*, nunca entre ellas.

In [0]:
# ── Hueco 1 · has_cr_card, nunca examinada ────────────────────────────────
print("— has_cr_card —")
print(df.groupby("has_cr_card")["exited"]
        .agg(abandono=lambda s: round(s.mean()*100, 1), n="size").to_string())

# Comprobar también que no esconde estructura dentro de otros niveles
print("\n— has_cr_card dentro de cada tramo de edad —")
print(df.pivot_table(index="age_group", columns="has_cr_card",
                     values="exited", aggfunc="mean").round(3).to_string())

— has_cr_card —
             abandono     n
has_cr_card                
False            20.8  2945
True             20.2  7055

— has_cr_card dentro de cada tramo de edad —
has_cr_card  False  True 
age_group                
18-29        0.075  0.076
30-39        0.102  0.112
40-49        0.307  0.308
50-59        0.587  0.548
60+          0.281  0.279


In [0]:
# ── Hueco 2 · ¿el efecto de la actividad es homogeneo? ────────────────────
print("— abandono por tramo de edad y actividad —")
tab = df.pivot_table(index="age_group", columns="is_active_member",
                     values="exited", aggfunc=["mean", "size"]).round(3)
print(tab.to_string())

print("\n— brecha activo/inactivo dentro de cada tramo —")
brecha = (df.groupby(["age_group", "is_active_member"])["exited"].mean()
            .unstack() * 100).round(1)
brecha.columns = ["inactivo", "activo"]
brecha["brecha_pp"] = (brecha["inactivo"] - brecha["activo"]).round(1)
print(brecha.sort_values("brecha_pp", ascending=False).to_string())

— abandono por tramo de edad y actividad —
                   mean         size      
is_active_member  False  True  False True 
age_group                                 
18-29             0.098  0.054   804   837
30-39             0.136  0.082  2165  2181
40-49             0.380  0.226  1396  1222
50-59             0.812  0.371   373   496
60+               0.856  0.125   111   415

— brecha activo/inactivo dentro de cada tramo —
           inactivo  activo  brecha_pp
age_group                             
60+            85.6    12.5       73.1
50-59          81.2    37.1       44.1
40-49          38.0    22.6       15.4
30-39          13.6     8.2        5.4
18-29           9.8     5.4        4.4


In [0]:
# ── Hueco 3 · colinealidad ENTRE predictores, no contra el objetivo ───────
X = df[["credit_score", "age", "tenure", "balance",
        "num_of_products", "estimated_salary"]].copy()
X["is_active"]    = df["is_active_member"].astype(int)
X["has_card"]     = df["has_cr_card"].astype(int)
X["balance_zero"] = df["balance_zero"].astype(int)
X["es_aleman"]    = (df["geography"] == "Germany").astype(int)
X["es_mujer"]     = (df["gender"] == "Female").astype(int)

corr = X.corr(method="spearman")

pares = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
fuertes = pares[abs(pares) > 0.3].sort_values(key=abs, ascending=False).round(3)

print("— pares de predictores con |rho| > 0.3 —")
print(fuertes.to_string() if len(fuertes) else "  ninguno")

print("\n— matriz completa —")
print(corr.round(2).to_string())

— pares de predictores con |rho| > 0.3 —
balance          balance_zero      -0.853
balance_zero     es_aleman         -0.436
balance          es_aleman          0.372
num_of_products  balance_zero       0.369
balance          num_of_products   -0.317

— matriz completa —
                  credit_score   age  tenure  balance  num_of_products  estimated_salary  is_active  has_card  balance_zero  es_aleman  es_mujer
credit_score              1.00 -0.01    0.00     0.01             0.01              0.00       0.02     -0.00         -0.01       0.01      0.00
age                      -0.01  1.00   -0.01     0.03            -0.06             -0.00       0.04     -0.02         -0.04       0.06      0.03
tenure                    0.00 -0.01    1.00    -0.01             0.01              0.01      -0.03      0.02          0.02      -0.00     -0.02
balance                   0.01  0.03   -0.01     1.00            -0.32              0.01      -0.01     -0.01         -0.85       0.37     -0.01
num

### Utilidades de gráfico

Dos funciones reutilizables para no repetir código en cada figura. La línea de tasa base aparece
siempre: sin ella, una barra al 30% no dice si es mucho o poco.

In [ ]:
CHURN, SAFE, ACC, GREY = "#a8471f", "#2c5a72", "#8a5a08", "#78736a"
INK = "#1c1a17"                    # color de todo el texto anotado

# Caja blanca semitransparente. Es lo que permite escribir encima de una barra
# sin que el texto se pierda: el problema no era el color, era el fondo.
CAJA = dict(boxstyle="round,pad=0.28", facecolor="white", edgecolor="none", alpha=0.82)


def linea_base(ax, lado="izquierda", y=None):
    """Dibuja la linea de la tasa base y la etiqueta sobre fondo blanco.

    La etiqueta va en coordenadas mixtas: x en fraccion de los ejes, y en
    unidades de datos. Asi queda siempre pegada a la linea y dentro del marco,
    mida lo que mida el grafico.
    """
    y = TASA_BASE * 100 if y is None else y
    ax.axhline(y, color=GREY, ls="--", lw=1.2, zorder=1)
    x, ha = (0.012, "left") if lado == "izquierda" else (0.988, "right")
    # Va DEBAJO de la linea a proposito: encima chocaba con los porcentajes en
    # negrita, que siempre se dibujan justo sobre el extremo de cada barra.
    ax.text(x, y - 1.0, f"tasa base {TASA_BASE:.1%}",
            transform=ax.get_yaxis_transform(), color=INK, fontsize=8.5,
            ha=ha, va="top", bbox=CAJA, zorder=6)


def nota(ax, texto, xy, xytext, ha="left"):
    """Anotacion con flecha. Texto en tinta oscura sobre caja blanca: legible
    caiga donde caiga, tambien encima de una barra naranja."""
    ax.annotate(texto, xy=xy, xytext=xytext, fontsize=8.5, color=INK,
                ha=ha, va="center", bbox=CAJA, zorder=7,
                arrowprops=dict(arrowstyle="->", color=CHURN, lw=1.2,
                                shrinkA=2, shrinkB=4))


def barras_abandono(serie_pct, serie_n, titulo, xlabel="", anotar_n=True,
                    ax=None, lado_base="izquierda"):
    """Barras de tasa de abandono con linea de tasa base y tamano muestral."""
    if ax is None:
        _, ax = plt.subplots()
    x = np.arange(len(serie_pct))
    colores = [CHURN if v >= TASA_BASE * 100 else SAFE for v in serie_pct]
    ax.bar(x, serie_pct, color=colores, width=0.62, zorder=2)
    linea_base(ax, lado_base)
    for i, (v, n) in enumerate(zip(serie_pct, serie_n)):
        ax.text(i, v + 1.5, f"{v:.1f}%", ha="center", fontsize=9,
                fontweight="bold", color=INK, zorder=5)
        if anotar_n:
            ax.text(i, 1.5, f"n={n:,}", ha="center", fontsize=7.5,
                    color="white", zorder=5)
    ax.set_title(titulo, fontsize=11.5, fontweight="bold", loc="left")
    ax.set_xticks(x); ax.set_xticklabels(serie_pct.index, fontsize=9)
    ax.set_ylabel("% abandono"); ax.set_xlabel(xlabel)
    ax.set_ylim(0, max(serie_pct) * 1.24)
    return ax


def mapa_calor(matriz, titulo, fmt="{:.2f}", cmap="RdBu_r", vlim=None, ax=None, cbar_label=""):
    """Mapa de calor anotado, sin dependencias externas."""
    if ax is None:
        _, ax = plt.subplots(figsize=(min(1.05 * matriz.shape[1] + 3, 12),
                                      0.55 * matriz.shape[0] + 2.2))
    v = vlim if vlim else np.nanmax(np.abs(matriz.values))
    im = ax.imshow(matriz.values, cmap=cmap, vmin=-v, vmax=v, aspect="auto")
    ax.set_xticks(range(matriz.shape[1])); ax.set_xticklabels(matriz.columns, rotation=45, ha="right", fontsize=8.5)
    ax.set_yticks(range(matriz.shape[0])); ax.set_yticklabels(matriz.index, fontsize=8.5)
    for i in range(matriz.shape[0]):
        for j in range(matriz.shape[1]):
            val = matriz.values[i, j]
            if np.isnan(val):
                continue
            ax.text(j, i, fmt.format(val), ha="center", va="center", fontsize=8,
                    color="white" if abs(val) > v * 0.55 else INK)
    ax.set_title(titulo, fontsize=11.5, fontweight="bold", loc="left")
    ax.grid(False)
    plt.colorbar(im, ax=ax, shrink=0.75, label=cbar_label)
    return ax


print("utilidades listas")

### Figura 1 · Colinealidad entre predictores

La matriz impresa como texto es ilegible. En mapa de calor, la estructura salta a la vista.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 7))
mapa_calor(corr, "Correlación de Spearman entre predictores (no contra el objetivo)",
           vlim=1.0, ax=ax, cbar_label="rho")
plt.tight_layout(); plt.show()

print("Pares relevantes:")
print(f"  balance <-> balance_zero   {corr.loc['balance','balance_zero']:+.3f}  (una deriva de la otra)")
print(f"  balance_zero <-> es_aleman {corr.loc['balance_zero','es_aleman']:+.3f}  (Alemania no tiene ceros)")
print(f"  is_active <-> resto        max {corr['is_active'].drop('is_active').abs().max():.3f}  (ortogonal)")

**Lo que muestra la figura**

Un bloque rojo/azul en la esquina inferior: `balance`, `balance_zero`, `num_of_products` y
`es_aleman` forman un grupo entrelazado. `balance` y `balance_zero` llegan a **−0,853**, lo
esperable cuando una se deriva de la otra.

*Consecuencia para el modelo:* en la regresión logística esas dos variables competirán por
explicar lo mismo, con coeficientes inestables. Los árboles no se ven afectados. Es material para
el feature selection del notebook 04.

**Y lo que muestra por ausencia:** `is_active_member` no correlaciona con nada — máximo 0,04
contra cualquier otro predictor. Es **ortogonal al resto del dataset**, así que toda su capacidad
predictiva es información nueva, no redundante. Justo lo contrario de `balance_zero`.

---

## B · La tesis: tres relaciones fuertes y no monótonas

> En este dataset, seleccionar variables por correlación habría descartado el mejor predictor.

Las tres figuras siguientes comparan lo que dice el coeficiente con lo que muestran los datos.

In [ ]:
tab_age = (df.groupby("age_group")
             .agg(pct=("exited", lambda s: s.mean()*100), n=("exited", "size"))
             .reindex(["18-29","30-39","40-49","50-59","60+"]))

ax = barras_abandono(tab_age["pct"], tab_age["n"],
                     "Figura 2 · Abandono por tramo de edad — U invertida",
                     xlabel="tramo de edad")
# La nota va sobre el hueco que dejan los dos primeros tramos, no sobre la barra
nota(ax, "el pico: más de la mitad\nde este tramo abandona",
     xy=(2.72, 56.0), xytext=(0.62, 47))
plt.tight_layout(); plt.show()

rho_age = df["age"].corr(df["exited"].astype(int), method="spearman")
print(f"Spearman age vs exited: {rho_age:+.4f}")
print(f"Recorrido real: {tab_age['pct'].min():.1f}% a {tab_age['pct'].max():.1f}%  "
      f"({tab_age['pct'].max() - tab_age['pct'].min():.1f} puntos)")

In [ ]:
tab_prod = (df.groupby("num_of_products")
              .agg(pct=("exited", lambda s: s.mean()*100), n=("exited", "size")))

ax = barras_abandono(tab_prod["pct"], tab_prod["n"],
                     "Figura 3 · Abandono por número de productos — el predictor que la correlación esconde",
                     xlabel="número de productos contratados")
# Hueco libre encima de los dos primeros grupos
nota(ax, "los 60 clientes con cuatro\nproductos se fueron todos",
     xy=(2.9, 100.0), xytext=(0.45, 92))
plt.tight_layout(); plt.show()

rho_prod = df["num_of_products"].corr(df["exited"].astype(int), method="spearman")
print(f"Spearman num_of_products vs exited: {rho_prod:+.4f}   <-- debil y NEGATIVA")
print(f"Recorrido real: {tab_prod['pct'].min():.1f}% a {tab_prod['pct'].max():.1f}%  "
      f"({tab_prod['pct'].max() - tab_prod['pct'].min():.1f} puntos)")

In [ ]:
# Figura 4 · lo que dice la correlación frente a lo que muestran los datos
vars_comp = ["age", "num_of_products", "balance", "credit_score",
             "estimated_salary", "tenure"]
filas = []
for v in vars_comp:
    rho = df[v].corr(df["exited"].astype(int), method="spearman")
    q = pd.qcut(df[v], 5, duplicates="drop") if df[v].nunique() > 10 else df[v]
    recorrido = df.groupby(q, observed=True)["exited"].mean()
    filas.append({"variable": v, "|rho|": abs(rho),
                  "recorrido_pp": (recorrido.max() - recorrido.min()) * 100})
comp = pd.DataFrame(filas).set_index("variable").sort_values("recorrido_pp", ascending=True)

fig, ax = plt.subplots(figsize=(9, 4.2))
y = np.arange(len(comp))
ax.barh(y - 0.19, comp["|rho|"] * 100, height=0.36, color=GREY, label="|Spearman| x 100")
ax.barh(y + 0.19, comp["recorrido_pp"], height=0.36, color=CHURN, label="recorrido real (pp entre grupos)")
ax.set_yticks(y); ax.set_yticklabels(comp.index, fontsize=9)
ax.set_title("Figura 4 · Lo que dice el coeficiente frente a lo que hacen los datos",
             fontsize=11.5, fontweight="bold", loc="left")
ax.set_xlabel("magnitud"); ax.legend(fontsize=8.5, loc="lower right")
for i, (r, rec) in enumerate(zip(comp["|rho|"] * 100, comp["recorrido_pp"])):
    ax.text(rec + 1.2, i + 0.19, f"{rec:.0f} pp", va="center", fontsize=8.5, color=CHURN)
    ax.text(r + 1.2, i - 0.19, f"{r/100:.3f}", va="center", fontsize=8, color=GREY)
plt.tight_layout(); plt.show()

print(comp.round(3).to_string())

**Interpretación de las figuras 2 a 4**

`num_of_products` recorre **92 puntos** entre grupos (del 7,6% al 100%) y su correlación de
Spearman es **−0,125** — débil y de signo contrario al efecto real. `age` recorre 48 puntos con un
coeficiente de +0,324.

La razón es la misma en ambos casos: **la correlación mide relación monótona**, y ninguna de las
dos lo es. Dos productos es el mínimo riesgo y cuatro el máximo, con uno en medio; la edad sube y
vuelve a bajar.

*Consecuencia práctica:* un filtro por `|rho| > 0.2` —criterio habitual— habría eliminado
`num_of_products`, que es el predictor más determinante del dataset.

*Consecuencia para el modelado:* los árboles particionan el espacio y capturan estas formas sin
ayuda. Un modelo lineal necesita las variables agrupadas (`age_group`, `products_group`) para
poder representarlas. Es la predicción, hecha antes de entrenar, de que Random Forest superará a
la regresión logística.

---

## C · El hallazgo principal: la interacción edad × actividad

La sección A destapó algo que obliga a revisar la tesis anterior.

In [ ]:
piv = (df.pivot_table(index="age_group", columns="is_active_member",
                      values="exited", aggfunc="mean")
         .reindex(["18-29","30-39","40-49","50-59","60+"]) * 100)
piv.columns = ["inactivo", "activo"]

tam = (df.pivot_table(index="age_group", columns="is_active_member",
                      values="exited", aggfunc="size")
         .reindex(["18-29","30-39","40-49","50-59","60+"]))
tam.columns = ["inactivo", "activo"]

fig, ax = plt.subplots(figsize=(7.5, 4.6))
im = ax.imshow(piv.values, cmap="Reds", vmin=0, vmax=100, aspect="auto")
ax.set_xticks([0, 1]); ax.set_xticklabels(["inactivo", "activo"], fontsize=10)
ax.set_yticks(range(5)); ax.set_yticklabels(piv.index, fontsize=9.5)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v, n = piv.values[i, j], tam.values[i, j]
        ax.text(j, i - 0.1, f"{v:.1f}%", ha="center", va="center", fontsize=12,
                fontweight="bold", color="white" if v > 50 else "#1c1a17")
        ax.text(j, i + 0.22, f"n={n:,}", ha="center", va="center", fontsize=8,
                color="white" if v > 50 else GREY)
ax.set_title("Figura 5 · Abandono por tramo de edad y actividad",
             fontsize=11.5, fontweight="bold", loc="left")
ax.grid(False)
plt.colorbar(im, ax=ax, shrink=0.8, label="% abandono")
plt.tight_layout(); plt.show()

In [ ]:
# Figura 6 · las dos curvas de edad, separadas por actividad
fig, ax = plt.subplots(figsize=(9, 4.8))
x = np.arange(5)
ax.plot(x, piv["inactivo"], "o-", color=CHURN, lw=2.4, ms=8, label="inactivos", zorder=3)
ax.plot(x, piv["activo"],   "o-", color=SAFE,  lw=2.4, ms=8, label="activos",   zorder=3)

# La tasa base va a la izquierda, donde las dos curvas estan muy por debajo:
# a la derecha se solapaba con la flecha de los 73 puntos
linea_base(ax, "izquierda")

for i, (a, b) in enumerate(zip(piv["inactivo"], piv["activo"])):
    ax.annotate("", xy=(i, a), xytext=(i, b),
                arrowprops=dict(arrowstyle="<->", color=GREY, lw=0.9, alpha=0.6))
    ax.text(i + 0.08, (a + b) / 2, f"{a-b:.0f} pp", fontsize=8, color=INK,
            va="center", bbox=CAJA, zorder=5)

nota(ax, "entre los inactivos el riesgo\nno deja de subir: no hay U invertida",
     xy=(3.92, 84.6), xytext=(2.30, 67))

ax.set_xticks(x); ax.set_xticklabels(piv.index)
ax.set_ylabel("% abandono"); ax.set_xlabel("tramo de edad")
ax.set_title("Figura 6 · La U invertida solo existe entre los clientes activos",
             fontsize=11.5, fontweight="bold", loc="left")
ax.legend(fontsize=9.5, loc="upper left", framealpha=0.9)
ax.set_ylim(0, 100); ax.set_xlim(-0.35, 4.45)
plt.tight_layout(); plt.show()

print("inactivos :", " -> ".join(f"{v:.1f}" for v in piv["inactivo"]), "  monótona creciente")
print("activos   :", " -> ".join(f"{v:.1f}" for v in piv["activo"]),   "  U invertida")
print(f"\n% de activos por tramo (composición):")
print((df.groupby("age_group")["is_active_member"].mean() * 100).round(1)
        .reindex(["18-29","30-39","40-49","50-59","60+"]).to_string())

**El tercer confusor, y el de mayores consecuencias**

Dábamos por establecido que la edad describe una U invertida: el riesgo sube hasta los 50-59 y
**cae** a partir de los 60. Separando por actividad:

```
activos    :  5,4 → 8,2 → 22,6 → 37,1 → 12,5     U invertida
inactivos  :  9,8 → 13,6 → 38,0 → 81,2 → 85,6    monótona creciente
```

**Entre los inactivos no hay ninguna U invertida.** El riesgo crece con la edad hasta el final, y
el 85,6% del tramo 60+ es la celda de mayor riesgo de todo el dataset.

La caída aparente en los mayores era un **artefacto de composición**: el 78,9% de ese tramo está
activo, frente a un 47-57% en el resto. Al agregar, esa mayoría de bajo riesgo arrastra la media.

*Por qué importa más que los otros dos confusores:* los anteriores nos evitaron creer un hallazgo
falso. Este **revela uno que no habíamos visto**, y afecta directamente a la recomendación de
negocio — que es el entregable final.

*Y para el modelado:* ninguna regresión logística sin término de interacción puede representar
esto. Los árboles sí, porque una rama parte primero por actividad y luego por edad. Refuerza la
predicción de la sección B.

In [ ]:
# Figura 7 · comparación de reglas prácticas frente a la línea base pre-registrada
CAPACIDAD = 800

def evaluar(mascara, nombre, cupo=CAPACIDAD):
    sub = df[mascara]
    if len(sub) > cupo:                      # muestreo aleatorio dentro de la regla
        tasa = sub["exited"].mean()
        contactos, captados = cupo, cupo * tasa
    else:
        contactos, captados = len(sub), sub["exited"].sum()
        resto = cupo - contactos
        if resto > 0:                        # rellenar con el resto de la base
            fuera = df[~mascara]
            contactos += resto
            captados  += resto * fuera["exited"].mean()
    return {"regla": nombre, "contactos": int(contactos), "captados": int(captados),
            "precision_pct": round(captados / contactos * 100, 1),
            "pct_abandono": round(captados / N_CHURN * 100, 1)}

reglas = [
    evaluar(df["age_group"].isin(["40-49", "50-59"]), "Edad 40-59 (pre-registrada)"),
    evaluar(~df["is_active_member"], "Solo inactivos"),
    evaluar(~df["is_active_member"] & df["age_group"].isin(["40-49","50-59","60+"]),
            "Inactivos de 40+"),
]
res = pd.DataFrame(reglas).set_index("regla")

fig, ax = plt.subplots(figsize=(9.5, 3.8))
y = np.arange(len(res))
ax.barh(y, res["captados"], color=[GREY, ACC, CHURN], height=0.55, zorder=2)

# Las etiquetas se alinean en columna, pasada la barra mas larga. Puestas junto
# a cada barra, la de 'Solo inactivos' cruzaba la linea de la base y se leia mal
x_txt = res["captados"].max() * 1.07
for i, (c, p) in enumerate(zip(res["captados"], res["precision_pct"])):
    ax.text(x_txt, i, f"{c} abandonos · precisión {p}%",
            va="center", ha="left", fontsize=9, color=INK, zorder=5)

base = res["captados"].iloc[0]
ax.axvline(base, color=GREY, ls="--", lw=1.2, zorder=1)
ax.text(base, len(res) - 0.42, "línea base pre-registrada", color=INK, fontsize=8.5,
        ha="center", va="bottom", bbox=CAJA, zorder=6)

ax.set_yticks(y); ax.set_yticklabels(res.index, fontsize=9)
ax.set_xlabel(f"abandonos captados con {CAPACIDAD} contactos")
ax.set_title("Figura 7 · El EDA produce una regla práctica mejor que la línea base",
             fontsize=11.5, fontweight="bold", loc="left")
ax.set_xlim(0, res["captados"].max() * 1.62)
ax.set_ylim(-0.55, len(res) - 0.15)
plt.tight_layout(); plt.show()

print(res.to_string())

**El listón sube**

El criterio pre-registrado en el notebook 00 se mantiene: con 800 contactos, el modelo debe captar
más de **448** abandonos. Cambiarlo ahora sería mover la portería.

Pero el propio EDA ha producido una regla práctica de dos variables claramente mejor. Se reportan
ambas, y el modelo tendrá que superar la buena para justificar su existencia. **Endurecer la
prueba después de registrarla no es hacer trampa — lo sería relajarla.**

---

## D · Las variables sensibles y sus efectos inexplicados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))

tab_geo = (df.groupby("geography")
             .agg(pct=("exited", lambda s: s.mean()*100), n=("exited", "size"))
             .sort_values("pct", ascending=False))
barras_abandono(tab_geo["pct"], tab_geo["n"],
                "Figura 8 · Abandono por país", ax=axes[0], lado_base="izquierda")

# Figura 9 con el MISMO formato que la 8: porcentaje con simbolo, en negrita,
# tamano muestral dentro de la barra y la tasa base etiquetada. Antes solo esta
# llevaba el numero pelado y las dos figuras no se leian igual.
gg = (df.pivot_table(index="geography", columns="gender", values="exited", aggfunc="mean") * 100)
nn = df.pivot_table(index="geography", columns="gender", values="exited", aggfunc="size")

ax = axes[1]
x = np.arange(len(gg))
ax.bar(x - 0.19, gg["Female"], width=0.36, color=CHURN, label="mujeres", zorder=2)
ax.bar(x + 0.19, gg["Male"],   width=0.36, color=SAFE,  label="hombres", zorder=2)
linea_base(ax, "derecha")

for i in range(len(gg)):
    for desp, col in [(-0.19, "Female"), (0.19, "Male")]:
        ax.text(i + desp, gg[col].iloc[i] + 0.8, f"{gg[col].iloc[i]:.1f}%",
                ha="center", fontsize=8.5, fontweight="bold", color=INK, zorder=5)
        ax.text(i + desp, 1.2, f"n={nn[col].iloc[i]:,}", ha="center",
                fontsize=7, color="white", zorder=5)

ax.set_xticks(x); ax.set_xticklabels(gg.index)
ax.set_ylabel("% abandono"); ax.set_xlabel("")
ax.set_title("Figura 9 · Género dentro de cada país", fontsize=11.5,
             fontweight="bold", loc="left")
ax.set_ylim(0, gg.values.max() * 1.24)
ax.legend(fontsize=9, loc="upper left", framealpha=0.9)
plt.tight_layout(); plt.show()

print("Perfil comparado — ninguna variable observable difiere:")
print(df.groupby("geography").agg(
    edad=("age","mean"), pct_activos=("is_active_member","mean"),
    productos=("num_of_products","mean"), abandono=("exited","mean")).round(3).to_string())

**Efectos reales, sin explicación disponible**

Alemania abandona al doble que Francia y España siendo **indistinguible en todas las variables
observables**: edad 39,8 frente a 38,5 y 38,9; actividad 49,7% frente a 51,7% y 53,0%; productos
idénticos hasta el segundo decimal. Comparando solo clientes con saldo, Alemania sigue trece
puntos por encima.

El género repite el patrón: +8,6 puntos globales, replicado en los tres países (+7,6 / +9,8 /
+8,1), sin que ninguna otra variable difiera.

Dos lecturas, y ambas van al informe:

- **De negocio:** existe un factor no capturado — competencia local, regulación, producto. La
  recomendación es recoger esos datos.
- **Metodológica:** que ninguna variable observable difiera y solo difiera el resultado es la
  huella de un generador sintético que asignó probabilidades distintas por grupo. Es la
  explicación más sencilla dado todo lo demás que hemos encontrado.

*Consecuencia:* `geography` y `gender` funcionarán como **sustitutos de algo desconocido**. El
modelo aprenderá «mujer alemana → riesgo alto» sin ninguna variable de comportamiento que lo
respalde. De ahí que el notebook 04 entrene dos versiones y mida el coste de excluirlas.

---

## E · Análisis inferenciales

La rúbrica pide un mínimo de dos. Se eligen deliberadamente uno que **debería rechazar** la
hipótesis nula y otro que **no debería** — reportar una ausencia de asociación con evidencia
formal dice más del criterio que acumular tests que confirman lo que uno espera.

In [ ]:
from scipy import stats


def chi2_test(var, nombre):
    tabla = pd.crosstab(df[var], df["exited"])
    chi2, p, gl, esp = stats.chi2_contingency(tabla)
    n_tot = tabla.values.sum()
    cramer = np.sqrt(chi2 / (n_tot * (min(tabla.shape) - 1)))
    print(f"— {nombre} —")
    print(f"  chi2 = {chi2:8.2f}   gl = {gl}   p = {p:.3e}")
    print(f"  V de Cramer = {cramer:.4f}   frecuencia esperada mínima = {esp.min():.1f}")
    print(f"  {'RECHAZA' if p < 0.05 else 'NO RECHAZA'} la independencia (alfa = 0.05)\n")
    return {"variable": nombre, "chi2": round(chi2, 2), "p": p,
            "cramer_v": round(cramer, 4), "rechaza": p < 0.05}

print("Prueba 1 · tenure — se espera NO rechazar")
r1 = chi2_test("tenure", "tenure vs exited")

print("Prueba 2 · is_active_member — se espera rechazar con claridad")
r2 = chi2_test("is_active_member", "is_active_member vs exited")

print("Complementaria · num_of_products")
r3 = chi2_test("products_group", "products_group vs exited")

In [ ]:
# Contraste no paramétrico para la edad: la distribución difiere entre quienes se van y quienes no
a = df.loc[df["exited"],  "age"]
b = df.loc[~df["exited"], "age"]
u, p_u = stats.mannwhitneyu(a, b, alternative="two-sided")

# Tamaño del efecto (correlación biserial por rangos)
r_rb = 1 - (2 * u) / (len(a) * len(b))

print("— Mann-Whitney U · edad entre quienes abandonan y quienes permanecen —")
print(f"  mediana abandonan  : {a.median():.0f} años   (n = {len(a):,})")
print(f"  mediana permanecen : {b.median():.0f} años   (n = {len(b):,})")
print(f"  U = {u:,.0f}   p = {p_u:.3e}")
print(f"  correlación biserial por rangos = {r_rb:+.4f}")
print(f"  {'RECHAZA' if p_u < 0.05 else 'NO RECHAZA'} la igualdad de distribuciones")

resumen = pd.DataFrame([r1, r2, r3]).set_index("variable")
print("\n— resumen —")
print(resumen.to_string())

**Interpretación**

La prueba sobre `tenure` **no rechaza** la independencia: no hay evidencia de asociación entre
antigüedad y abandono. Es un resultado negativo, y se reporta como tal. Refuerza además la
conclusión de que la variable procede de un sorteo uniforme.

Las pruebas sobre `is_active_member` y `products_group` rechazan con holgura. Pero conviene mirar
la **V de Cramér** y no solo el p-valor: con 10.000 observaciones, casi cualquier diferencia
resulta significativa. La V mide el *tamaño* del efecto, que es lo que importa para decidir si
vale la pena actuar.

> **Advertencia obligada:** todo esto son **asociaciones**. Ningún test de independencia establece
> causalidad. Que la inactividad se asocie al abandono no prueba que reactivar lo evite — podría
> ser síntoma y no causa: quien ya decidió irse deja de usar la cuenta. Distinguirlo exige un
> experimento, no este dataset.

---

## F · Qué podría habérsenos escapado

La sección A demostró que la pregunta no es retórica: **`has_cr_card` llevaba todo el proyecto sin
mirarse**, y la interacción edad × actividad —el hallazgo principal de este notebook— apareció
justo al tapar ese tipo de hueco.

### Cobertura alcanzada

| Comprobación | Estado |
|---|---|
| Cada variable contra el objetivo, **por grupos** | completa |
| Cruces con los cuatro predictores dominantes | completa |
| Colinealidad entre predictores | medida (figura 1) |
| Interacciones de tres o más variables | **no exploradas** |
| Efectos no lineales dentro de subgrupos pequeños | **parcialmente** |

### Lo que queda sin cubrir, y quién lo detectará

Las dos últimas filas no se cubren aquí a propósito: el espacio de interacciones triples es
demasiado grande para recorrerlo a mano, y explorarlo a ojo invita a encontrar patrones donde solo
hay ruido.

**El notebook 04 hace de auditor.** Un Random Forest recorre ese espacio sin que nadie le indique
dónde mirar. Tres señales obligan a volver a este notebook:

1. **Importancia alta en una variable que aquí descartamos** — `credit_score`, `tenure`,
   `estimated_salary` o `has_cr_card`. Si alguna aparece arriba, se nos escapó algo.
2. **Un corte temprano por una variable que no discutimos.**
3. **Rendimiento muy por encima de lo previsto.** Con lo que sabemos, el ROC-AUC esperable ronda
   0,85. Bastante más que eso significaría estructura no detectada — o fuga de datos.

### Predicciones registradas antes de modelar

Para poder comprobarlas después:

| # | Predicción |
|---|---|
| P1 | Random Forest superará a la regresión logística — las relaciones clave no son monótonas |
| P2 | Las tres variables más importantes serán `age`, `num_of_products` e `is_active_member` |
| P3 | `credit_score`, `tenure`, `estimated_salary` y `has_cr_card` quedarán al final |
| P4 | ROC-AUC entre 0,82 y 0,88 |
| P5 | Excluir `gender` y `geography` costará entre 2 y 5 puntos de recall |

---

## Resumen del EDA

**Tesis confirmada.** Las tres relaciones más fuertes son no monótonas y en las tres la
correlación engaña — `num_of_products` recorre 92 puntos con un coeficiente de −0,125.

**Hallazgo principal.** La interacción edad × actividad. Entre los inactivos el riesgo crece de
forma monótona hasta el 85,6% en los mayores de 60; la U invertida solo existe entre los activos y
era un artefacto de composición.

**Tercer confusor del proyecto.** Los dos anteriores desmontaron hallazgos falsos; este reveló uno
que no habíamos visto.

**Efectos sin explicación.** Alemania y el género predicen con fuerza sin que ninguna variable
observable los respalde. Van al análisis de sesgo.

**Ausencias documentadas.** `tenure`, `credit_score`, `estimated_salary` y `has_cr_card` no tienen
señal. Confirmado por grupos y, en el caso de `tenure`, con una prueba formal que no rechaza.

**El listón subió.** El EDA produjo una regla práctica de dos variables mejor que la línea base
pre-registrada. Se reportan ambas.

**Siguiente:** `04_modelado` — split estratificado, cuatro modelos, dos versiones para medir el
coste de las variables sensibles, y la auditoría de las cinco predicciones registradas arriba.